In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


tickers = ["JNJ", "AAPL", "MSFT", "PG", "KO"]
start = "2017-01-01"
end = "2020-12-31"

stock_data = {}

/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Download market indext data for S&P 500 index ticker
sp500 = yf.download("^GSPC", start=start, end=end, auto_adjust=True, progress=False)
sp500.columns = sp500.columns.droplevel(1)
sp500_market_return = sp500['Close'].pct_change().to_frame(name='market_return')
sp500_market_return

,market_return
Date,
2017-01-03,NaN
2017-01-04,0.005722
2017-01-05,-0.000771
2017-01-06,0.003517
2017-01-09,-0.003549
...,...
2020-12-23,0.000746
2020-12-24,0.003537
2020-12-28,0.008723


In [3]:
# Download stock data for each ticker and store it in a dictionary
for ticker in tickers:
    stock_data[ticker] = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    stock_data[ticker].columns = stock_data[ticker].columns.droplevel(1)
    stock_data[ticker] = stock_data[ticker].merge(sp500_market_return, left_index=True, right_index=True, how='inner')

df = stock_data['AAPL']
df

,Close,High,Low,Open,Volume,market_return
Date,,,,,,
2017-01-03,26.745855,26.787304,26.425780,26.665261,115127600,NaN
2017-01-04,26.715927,26.828761,26.653755,26.676782,84472400,0.005722
2017-01-05,26.851782,26.909349,26.667565,26.692895,88774400,-0.000771
2017-01-06,27.151134,27.208702,26.819545,26.890928,127007600,0.003517
2017-01-09,27.399818,27.501138,27.158036,27.160337,134247600,-0.003549
...,...,...,...,...,...,...
2020-12-23,127.364159,128.793782,127.189093,128.531207,88223700,0.000746
2020-12-24,128.346420,129.795514,127.500313,127.714274,54930100,0.003537
2020-12-28,132.936798,133.568945,129.844106,130.310937,124486200,0.008723


In [4]:
# Calculate stock returns variable
df['stock_return'] = df['Close'].pct_change()
# Target column is the stock return of the next day, so we shift the 'stock_return' column by -1
df['target'] = df['stock_return'].shift(-1)
df

,Close,High,Low,Open,Volume,market_return,stock_return,target
Date,,,,,,,,
2017-01-03,26.745855,26.787304,26.425780,26.665261,115127600,NaN,NaN,-0.001119
2017-01-04,26.715927,26.828761,26.653755,26.676782,84472400,0.005722,-0.001119,0.005085
2017-01-05,26.851782,26.909349,26.667565,26.692895,88774400,-0.000771,0.005085,0.011148
2017-01-06,27.151134,27.208702,26.819545,26.890928,127007600,0.003517,0.011148,0.009159
2017-01-09,27.399818,27.501138,27.158036,27.160337,134247600,-0.003549,0.009159,0.001009
...,...,...,...,...,...,...,...,...
2020-12-23,127.364159,128.793782,127.189093,128.531207,88223700,0.000746,-0.006976,0.007712
2020-12-24,128.346420,129.795514,127.500313,127.714274,54930100,0.003537,0.007712,0.035766
2020-12-28,132.936798,133.568945,129.844106,130.310937,124486200,0.008723,0.035766,-0.013315


In [5]:
def add_moving_average(df, n, price_col='Close'):
    """
    Calculate n-day Simple Moving Average (SMA) on a financial time series.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing price data
    n : int
        Window size (e.g., 10, 50, 200)
    price_col : str
        Column name to calculate SMA on (default: 'Close')

    Returns:
    --------
    df : pandas.DataFrame
        DataFrame with new SMA column added
    """
    
    df[f'SMA_{n}'] = df[price_col].rolling(window=n, min_periods=n).mean()
    return df

def add_volatility(df, n=20, price_col='Close', annualize=False):
    """
    Add an n-day rolling volatility feature based on past returns.

    This is a rolling-window feature engineering step:
    - returns describe day-to-day price changes
    - rolling std of returns estimates recent noise / uncertainty
    - if annualize=True, daily volatility is scaled by sqrt(252)

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing price data.
    n : int
        Rolling window size.
    price_col : str, default='Close'
        Column used to compute returns.
    annualize : bool, default=False
        If True, annualise daily volatility using sqrt(252).

    Returns
    -------
    pandas.DataFrame
        Copy of df with:
        - 'return'
        - f'volatility_{n}' (or annualised version)
    """

    # Rolling volatility = recent standard deviation of returns
    vol_col = f'volatility_{n}'
    df[vol_col] = df['stock_return'].rolling(window=n, min_periods=n).std()

    # Optional annualisation for finance applications
    if annualize:
        df[vol_col] = df[vol_col] * np.sqrt(252)

    return df

def add_momentum_rsi(df, n=14, price_col='Close', col_name=None):
    """
    Adds Relative Strength Index (RSI) feature to dataframe.

    Parameters:
    - df: pandas DataFrame
    - n: lookback window (default = 14, standard in finance)
    - price_col: column name for price (default = 'Close')
    - col_name: optional custom column name

    Returns:
    - df with RSI column added
    """
    
    if col_name is None:
        col_name = f'RSI_{n}'
    
    # Price changes (returns in price space, not percentage)
    delta = df[price_col].diff()

    # Gains and losses
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)

    gain = pd.Series(gain, index=df.index)
    loss = pd.Series(loss, index=df.index)

    # Rolling averages (Wilder's smoothing approximation using mean)
    avg_gain = gain.rolling(window=n, min_periods=n).mean()
    avg_loss = loss.rolling(window=n, min_periods=n).mean()

    # Avoid division by zero
    rs = avg_gain / (avg_loss + 1e-10)

    # RSI formula
    rsi = 100 - (100 / (1 + rs))

    df[col_name] = rsi

    return df

In [6]:
add_moving_average(df,10)
add_moving_average(df,50)
add_moving_average(df,200)
add_volatility(df)
add_momentum_rsi(df)

,Close,High,Low,Open,Volume,market_return,stock_return,target,SMA_10,SMA_50,SMA_200,volatility_20,RSI_14
Date,,,,,,,,,,,,,
2017-01-03,26.745855,26.787304,26.425780,26.665261,115127600,NaN,NaN,-0.001119,NaN,NaN,NaN,NaN,NaN
2017-01-04,26.715927,26.828761,26.653755,26.676782,84472400,0.005722,-0.001119,0.005085,NaN,NaN,NaN,NaN,NaN
2017-01-05,26.851782,26.909349,26.667565,26.692895,88774400,-0.000771,0.005085,0.011148,NaN,NaN,NaN,NaN,NaN
2017-01-06,27.151134,27.208702,26.819545,26.890928,127007600,0.003517,0.011148,0.009159,NaN,NaN,NaN,NaN,NaN
2017-01-09,27.399818,27.501138,27.158036,27.160337,134247600,-0.003549,0.009159,0.001009,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-12-23,127.364159,128.793782,127.189093,128.531207,88223700,0.000746,-0.006976,0.007712,123.469108,116.173881,95.856817,0.016705,67.006043
2020-12-24,128.346420,129.795514,127.500313,127.714274,54930100,0.003537,0.007712,0.035766,124.318140,116.387622,96.198655,0.016705,70.334872
2020-12-28,132.936798,133.568945,129.844106,130.310937,124486200,0.008723,0.035766,-0.013315,125.706929,116.702490,96.527515,0.017922,73.857024
